# MAP: Charting Student Math Misunderstandings — Gold-Medal Pipeline

## Competition Overview
Predict students' **math misconceptions** from their open-ended explanations.  
Given `QuestionText + MC_Answer + StudentExplanation` → predict `Category:Misconception` (65 classes).  
Metric: **MAP@3** (Mean Average Precision at 3).  
Dataset: 36,696 training samples × 7 columns; only **15 unique QuestionIds** (critical for CV design).

## Pipeline Architecture — Two-Model Ensemble (Classification + Retrieval)

```
┌────────────────────────────────────────────────────────────────────────┐
│                      GOLD-MEDAL PIPELINE (H100)                        │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  MODEL A — DeBERTa-v3-large Classification (Primary)                  │
│    ├─ 65-class classification head                                     │
│    ├─ GroupKFold 5-fold × 2-seed × 4-epoch                            │
│    ├─ Focal Loss (γ=2) + inverse-frequency class weights              │
│    ├─ AWP + R-Drop + EMA + Layerwise LR Decay                        │
│    ├─ H100: batch=16, grad_accum=2 → eff. batch=32, bf16              │
│    └─ Family filter + learned temperature scaling                     │
│                                                                        │
│  MODEL B — BGE-Large Retrieval (Secondary)                            │
│    ├─ Fine-tune BAAI/bge-large-en-v1.5 (bi-encoder)                  │
│    ├─ MultipleNegativesRankingLoss (in-batch negatives)               │
│    ├─ Query  = question + answer + is_correct_flag + explanation      │
│    ├─ Corpus = enriched label descriptions (65 labels)                │
│    ├─ GroupKFold 5-fold × 3 epochs; best-fold checkpoint saved        │
│    └─ Cosine similarity → softmax (T=20) → retrieval probs            │
│                                                                        │
│  ENSEMBLE (A + B)                                                      │
│    ├─ α × DeBERTa_probs + (1-α) × Retrieval_probs                    │
│    ├─ α ∈ [0.0, 0.05, …, 1.0] — learned via OOF grid search          │
│    └─ Family filter + temperature scaling → Top-3                     │
│                                                                        │
│  POST-PROCESSING                                                       │
│    ├─ Family filter (True_*/False_* prefix constraint from is_correct) │
│    ├─ Learned temperature scaling — Guo et al., ICML 2017             │
│    └─ Ensemble weight α optimized on OOF                             │
│                                                                        │
│  EXPLAINABLE AI (Optional — skipped when QUICK_RUN=True)             │
│    ├─ Primary: Qwen2.5-7B-Instruct (4-bit NF4, BitsAndBytes)         │
│    └─ Fallback: google/flan-t5-large (seq2seq, full precision)        │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

## Key Insight: Why Classification Alone Fails

The DeBERTa classifier treats each of the 65 labels independently, ignoring semantic similarity between misconception types. Adding a **retrieval model** (Sentence Transformer) captures *semantic similarity* between student explanations and misconception descriptions, providing crucial diversity for the ensemble. Their errors are partially uncorrelated — DeBERTa excels at common/discriminative classes while retrieval generalizes better on rare/confusable ones.

> **Note**: Top Kaggle solutions on the full ~2,500-misconception Eedi task used two-stage retrieval + reranking (MAP@25). This pipeline adapts those ideas for our 65-class reformulation.

---

## References

| # | Technique | Paper |
|---|-----------|-------|
| 1 | **DeBERTa-v3** | He et al. (2023). *DeBERTaV3: Improving DeBERTa using ELECTRA-Style Pre-Training.* ICLR 2023. [arXiv:2111.09543](https://arxiv.org/abs/2111.09543) |
| 2 | **Sentence-BERT / Bi-Encoder** | Reimers & Gurevych (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP 2019. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084) |
| 3 | **BGE Embeddings** | Xiao et al. (2024). *C-Pack: Packaged Resources for General Chinese Embeddings.* SIGIR 2024. [arXiv:2309.07597](https://arxiv.org/abs/2309.07597) |
| 4 | **MultipleNegativesRankingLoss** | Henderson et al. (2017). *Efficient Natural Language Response Suggestion for Smart Reply.* [arXiv:1705.00652](https://arxiv.org/abs/1705.00652) |
| 5 | **Focal Loss** | Lin et al. (2017). *Focal Loss for Dense Object Detection.* ICCV 2017. [arXiv:1708.02002](https://arxiv.org/abs/1708.02002) |
| 6 | **AWP** | Wu et al. (2020). *Adversarial Weight Perturbation Helps Robust Generalization.* NeurIPS 2020. [arXiv:2004.05884](https://arxiv.org/abs/2004.05884) |
| 7 | **R-Drop** | Wu et al. (2021). *R-Drop: Regularized Dropout for Neural Networks.* NeurIPS 2021. [arXiv:2106.14448](https://arxiv.org/abs/2106.14448) |
| 8 | **EMA** | Polyak & Juditsky (1992). *Acceleration of Stochastic Approximation by Averaging.* SIAM J. Control and Optimization, 30(4), 838–855. |
| 9 | **Temperature Scaling** | Guo et al. (2017). *On Calibration of Modern Neural Networks.* ICML 2017. [arXiv:1706.04599](https://arxiv.org/abs/1706.04599) |
| 10 | **Layerwise LR Decay** | Clark et al. (2020). *ELECTRA: Pre-training Text Encoders as Discriminators.* ICLR 2020. [arXiv:2003.10555](https://arxiv.org/abs/2003.10555) |
| 11 | **Cosine Annealing w/ Warmup** | Loshchilov & Hutter (2017). *SGDR: Stochastic Gradient Descent with Warm Restarts.* ICLR 2017. [arXiv:1608.03983](https://arxiv.org/abs/1608.03983) |
| 12 | **Deep Ensembles** | Lakshminarayanan et al. (2017). *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles.* NeurIPS 2017. [arXiv:1612.01474](https://arxiv.org/abs/1612.01474) |
| 13 | **Label Smoothing** | Szegedy et al. (2016). *Rethinking the Inception Architecture for Computer Vision.* CVPR 2016. [arXiv:1512.00567](https://arxiv.org/abs/1512.00567) |
| 14 | **GroupKFold** | scikit-learn. Group-aware cross-validation that prevents leakage when the same QuestionId appears in both train and validation sets. |
| 15 | **Qwen2.5** | Qwen Team (2024). *Qwen2.5 Technical Report.* [arXiv:2412.15115](https://arxiv.org/abs/2412.15115) |
| 16 | **4-bit NF4 Quantization** | Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* NeurIPS 2023. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314) |

## Cell 1 — Environment Setup & Dependencies

In [ ]:
"""
Environment Setup — Run once, then restart kernel
"""
import sys, subprocess

def pip_install(cmd):
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd.split())

# Core ML
pip_install("install -U -q pip setuptools wheel")
pip_install("install -q -U transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 sentencepiece")
pip_install("install -q -U sentence-transformers>=2.7.0")

# ============================================================
# IMPORTS
# ============================================================
import os, gc, json, re, warnings, shutil, datetime
from pathlib import Path
from collections import Counter
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import normalize

from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding, set_seed,
    get_cosine_schedule_with_warmup,
)
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer, InputExample,
    losses, evaluation,
)
from torch.utils.data import DataLoader as TorchDataLoader

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    # Check for bf16 support (H100/A100)
    BF16_OK = torch.cuda.is_bf16_supported()
    print(f"bf16 supported: {BF16_OK}")
else:
    BF16_OK = False
print("All imports successful.")


## Cell 2 — Configuration (H100-Optimized)

All hyperparameters tuned for **NVIDIA H100 80 GB**. Key differences from T4:
- **4× larger effective batch size**: bf16 + 80 GB VRAM → `BATCH_SIZE=16 × GRAD_ACCUM=2` = 32
- **Full run** (`QUICK_RUN=False`): 5 folds × 2 seeds × 4 epochs — safe within 12-hr Kaggle limit
- **Quick test** (`QUICK_RUN=True`): 2 folds × 1 seed × 1 epoch, no AWP/R-Drop ≈ 10 min
- **Two-model ensemble**: DeBERTa-v3-large classifier + BAAI/bge-large-en-v1.5 retriever


In [ ]:
# ============================================================
# CONFIGURATION — H100-OPTIMIZED (80 GB VRAM, bf16)
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  QUICK_RUN TOGGLE — flip this to True for a smoke test  │
# │  Runs 1 fold × 1 seed × 1 epoch, no AWP/R-Drop         │
# │  Expected time: ~10 min on H100                         │
# │  Set to False for the full gold-medal submission run    │
# └─────────────────────────────────────────────────────────┘
QUICK_RUN = True   # ← change to True to test, False to submit

# --- Paths ---
# Kaggle paths (uncomment for Kaggle):
TRAIN_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
TEST_PATH  = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv"
# Local paths (uncomment for local):
# TRAIN_PATH = "DS/train.csv"
# TEST_PATH  = "DS/test.csv"
WORKDIR = Path("./")

# ╔══════════════════════════════════════════════════════════════════╗
# ║  MODEL A — DeBERTa-v3-large Classifier (Primary Model)         ║
# ╚══════════════════════════════════════════════════════════════════╝
CLASSIFIER_MODEL = "microsoft/deberta-v3-large"

# H100-optimized training settings (FULL RUN defaults)
N_FOLDS     = 5
SEEDS       = [42, 123]      # 2 seeds — safe within 12hr Kaggle limit
EPOCHS      = 4              # 4 epochs — sufficient convergence, saves ~20% time
BATCH_SIZE  = 16          # H100 can handle 16 for deberta-v3-large
GRAD_ACCUM  = 2           # Effective batch = 32
LR          = 1.5e-5
MAX_LEN     = 512
WEIGHT_DECAY = 0.02
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

# Mixed precision: bf16 on H100 (fp16 fallback)
USE_BF16 = BF16_OK
USE_FP16 = not BF16_OK

# --- Focal Loss (Lin et al., 2017) — better than label smoothing for imbalance ---
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0
USE_CLASS_WEIGHTS = True     # Inverse-frequency class weighting

# --- Label Smoothing (fallback if focal=False) ---
LABEL_SMOOTHING = 0.0        # Set >0 only if USE_FOCAL_LOSS=False

# --- Layerwise LR Decay (Clark et al., 2020) ---
USE_LAYERWISE_LR_DECAY = True
LAYERWISE_LR_DECAY = 0.95

# --- AWP (Wu et al., 2020) ---
USE_AWP = True
AWP_START_EPOCH = 1
AWP_LR  = 0.1
AWP_EPS = 1e-4

# --- R-Drop (Wu et al., 2021) ---
USE_RDROP = True
RDROP_ALPHA = 0.5

# --- EMA (Polyak & Juditsky, 1992) ---
USE_EMA = True
EMA_DECAY = 0.999

# --- Post-processing ---
USE_FAMILY_FILTER = True
USE_TEMPERATURE_SCALING = True
TEMPERATURE = 1.5

# --- Model saving ---
SAVE_FOLD_MODELS = True
MODEL_SAVE_DIR = WORKDIR / "fold_models"

# ╔══════════════════════════════════════════════════════════════════╗
# ║  MODEL B — Sentence Transformer Retriever (Secondary Model)     ║
# ╚══════════════════════════════════════════════════════════════════╝
RETRIEVER_MODEL = "BAAI/bge-large-en-v1.5"     # Top embedding model
RETRIEVER_EPOCHS = 3
RETRIEVER_BATCH = 48       # Reduced from 64 to ease peak memory
RETRIEVER_LR = 2e-5
RETRIEVER_WARMUP = 0.1
RETRIEVER_FOLDS = 5        # Same fold structure as classifier

# ╔══════════════════════════════════════════════════════════════════╗
# ║  ENSEMBLE — Combine DeBERTa + Sentence Transformer              ║
# ╚══════════════════════════════════════════════════════════════════╝
ENSEMBLE_ALPHA = None       # None = learn from OOF, float = fixed weight
# Final score = α × DeBERTa_score + (1-α) × Retrieval_score

# ╔══════════════════════════════════════════════════════════════════╗
# ║  EXPLAINABLE AI (Optional — LLM for teacher feedback)           ║
# ╚══════════════════════════════════════════════════════════════════╝
USE_LLM_EXPLAINER = True
EXPLAINER_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EXPLAINER_FALLBACK_MODEL = "google/flan-t5-large"
USE_LLM_VERIFIER = True
KB_BUILD_TOP_N = 200
KB_EXAMPLES_PER_MISCON = 6
KB_PATH = WORKDIR / "misconception_kb.json"

# ============================================================
# QUICK_RUN OVERRIDES — applied automatically, do not edit
# ============================================================
if QUICK_RUN:
    N_FOLDS          = 2          # Only 2 folds
    SEEDS            = [42]       # Single seed
    EPOCHS           = 1          # Single epoch
    RETRIEVER_EPOCHS = 1          # Single epoch for retriever
    RETRIEVER_FOLDS  = 2
    USE_AWP          = False      # Skip — adds 50% time per step
    USE_RDROP        = False      # Skip — doubles forward passes
    USE_LLM_EXPLAINER = False     # Skip — saves 20-30 min
    KB_BUILD_TOP_N   = 5          # Tiny KB for smoke test
    SAVE_FOLD_MODELS = False      # Skip disk writes

# ============================================================
# TIME ESTIMATE ON H100 (with R-Drop 2x forward + AWP):
#   QUICK_RUN=True : ~10 min  (smoke test only, score meaningless)
#   QUICK_RUN=False: ~5.5 hrs (safe within 12-hr Kaggle limit)
#     Model A: 5 folds × 2 seeds × 4 epochs × ~6min = ~4 hrs
#     Model B: 5 folds × 3 epochs × ~2min            = ~30 min
#     Inference + Ensemble                            = ~15 min
#     KB + Explainable AI                             = ~30 min
# ============================================================

_mode = "⚡ QUICK RUN (smoke test)" if QUICK_RUN else "🏆 FULL GOLD-MEDAL RUN"
print("=" * 60)
print(f"  MODE: {_mode}")
print("=" * 60)
print(f"  Classifier: {CLASSIFIER_MODEL}")
print(f"  Retriever:  {RETRIEVER_MODEL}")
print(f"  DeBERTa — Folds: {N_FOLDS} | Seeds: {SEEDS} | Epochs: {EPOCHS}")
print(f"  DeBERTa — Effective batch: {BATCH_SIZE * GRAD_ACCUM} | bf16: {USE_BF16}")
print(f"  Retriever — Folds: {RETRIEVER_FOLDS} | Epochs: {RETRIEVER_EPOCHS}")
print(f"  Focal Loss (γ={FOCAL_GAMMA}): {USE_FOCAL_LOSS}")
print(f"  AWP: {USE_AWP} | R-Drop: {USE_RDROP} | EMA: {USE_EMA}")
if QUICK_RUN:
    print("\n  ⚠  QUICK_RUN=True — results are NOT submission-quality!")

## Cell 3 — Data Loading & EDA


In [ ]:
# ============================================================
# DATA LOADING & EDA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
print(f"Dataset shape: {train_df.shape}")
print(f"Columns: {train_df.columns.tolist()}\n")

# Fill NaN
train_df["StudentExplanation"] = train_df["StudentExplanation"].fillna("")
train_df["QuestionText"]       = train_df["QuestionText"].fillna("")
train_df["MC_Answer"]          = train_df["MC_Answer"].fillna("")
train_df["Misconception"]      = train_df["Misconception"].fillna("NA")

# Create combined label
train_df["label_str"] = train_df["Category"] + ":" + train_df["Misconception"]

print(f"Unique Categories:     {train_df['Category'].nunique()}")
print(f"Unique Misconceptions: {train_df['Misconception'].nunique()}")
print(f"Unique Labels (target):{train_df['label_str'].nunique()}")
print(f"Unique Questions:      {train_df['QuestionId'].nunique()}")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Category distribution
cat_counts = train_df["Category"].value_counts()
ax = axes[0, 0]
bars = ax.barh(cat_counts.index, cat_counts.values, color=sns.color_palette("viridis", len(cat_counts)))
ax.set_title("Category Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Count")
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, f"{val:,}", va="center", fontsize=10)

# 2. Top-15 Misconceptions
miscon_counts = train_df[train_df["Misconception"] != "NA"]["Misconception"].value_counts().head(15)
ax = axes[0, 1]
ax.barh(miscon_counts.index[::-1], miscon_counts.values[::-1], color=sns.color_palette("magma", 15))
ax.set_title("Top 15 Misconceptions", fontsize=14, fontweight="bold")
ax.set_xlabel("Count")

# 3. Explanation length distribution
train_df["expl_len"] = train_df["StudentExplanation"].str.split().str.len().fillna(0).astype(int)
ax = axes[1, 0]
ax.hist(train_df["expl_len"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax.set_title("Student Explanation Length (words)", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Words")
ax.set_ylabel("Frequency")

# 4. Class imbalance visualization (top-20 labels)
label_counts = train_df["label_str"].value_counts()
ax = axes[1, 1]
top20 = label_counts.head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color=sns.color_palette("coolwarm", 20))
ax.set_title("Top 20 Labels — Heavy Imbalance!", fontsize=14, fontweight="bold")
ax.set_xlabel("Count")

plt.tight_layout()
plt.suptitle("MAP Dataset — EDA", fontsize=16, fontweight="bold", y=1.02)
plt.show()

# Summary
print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total samples:          {len(train_df):,}")
print(f"Unique labels:          {train_df['label_str'].nunique()}")
print(f"Class imbalance ratio:  {label_counts.iloc[0] / label_counts.iloc[-1]:.1f}x")
print(f"Top label:              {label_counts.index[0]} ({label_counts.iloc[0]:,}, {label_counts.iloc[0]/len(train_df)*100:.1f}%)")
print(f"Bottom label:           {label_counts.index[-1]} ({label_counts.iloc[-1]:,})")


## Cell 4 — Feature Engineering + Label Descriptions for Retrieval

Two parallel feature paths:
1. **Classification inputs**: structured `Question + Answer + Is_Correct + Explanation` for DeBERTa
2. **Retrieval corpus**: enriched natural-language descriptions of each of the 65 labels, used as the "document" side of the bi-encoder


In [ ]:
# ============================================================
# FEATURE ENGINEERING + LABEL DESCRIPTIONS
# ============================================================

# --- 4a. Infer correct answer per QuestionId ---
def build_correct_by_q(df: pd.DataFrame) -> dict:
    correct_mask = df["Category"].str.startswith("True", na=False)
    tmp = df.loc[correct_mask, ["QuestionId", "MC_Answer"]].dropna()
    if len(tmp) == 0:
        return {}
    return tmp.groupby("QuestionId")["MC_Answer"].agg(
        lambda s: s.value_counts().index[0]
    ).to_dict()

correct_by_q_full = build_correct_by_q(train_df)
print(f"Correct answers inferred for {len(correct_by_q_full)} questions")

# --- 4b. Compute is_correct flag ---
def compute_is_correct(df, correct_by_q):
    out = np.full(len(df), -1, dtype=np.int8)
    for i, (q, a) in enumerate(zip(df["QuestionId"].values, df["MC_Answer"].values)):
        if pd.isna(q) or pd.isna(a):
            continue
        if q in correct_by_q:
            out[i] = 1 if str(a) == str(correct_by_q[q]) else 0
    return out

train_df["is_correct"] = compute_is_correct(train_df, correct_by_q_full)

# --- 4c. Build structured input text ---
def make_text(df, is_correct):
    q  = df["QuestionText"].fillna("").astype(str)
    a  = df["MC_Answer"].fillna("").astype(str)
    e  = df["StudentExplanation"].fillna("").astype(str)
    ic = pd.Series(is_correct).map({1: "Yes", 0: "No"}).fillna("Unknown").astype(str)
    return (
        "Question: " + q +
        "\nAnswer: " + a +
        "\nIs Correct Answer: " + ic +
        "\nStudent Explanation: " + e
    ).tolist()

# --- 4d. Prepare label encoding ---
classes = np.array(sorted(train_df["label_str"].unique().tolist()))
lab2id = {c: i for i, c in enumerate(classes)}
id2lab = {i: c for c, i in lab2id.items()}
y_all  = train_df["label_str"].map(lab2id).astype(int).values
num_labels = len(classes)

# Precompute True/False masks for family filtering
true_mask  = np.array([c.startswith("True")  for c in classes], dtype=bool)
false_mask = np.array([c.startswith("False") for c in classes], dtype=bool)

# --- 4e. Compute class weights for Focal Loss ---
label_counts = train_df["label_str"].value_counts()
class_weights_np = np.zeros(num_labels, dtype=np.float32)
for lab_str, cnt in label_counts.items():
    idx = lab2id[lab_str]
    # Inverse frequency weighting, capped to avoid extreme weights
    class_weights_np[idx] = min(len(train_df) / (num_labels * cnt), 10.0)
# Normalize so mean=1
class_weights_np = class_weights_np / class_weights_np.mean()
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
print(f"Class weights: min={class_weights_np.min():.3f}, max={class_weights_np.max():.3f}, mean={class_weights_np.mean():.3f}")

# ==============================================================
# 4f. BUILD ENRICHED LABEL DESCRIPTIONS (for Sentence Transformer)
# ==============================================================
# Each label gets a rich natural-language description so that the
# bi-encoder can learn semantic similarity between student
# explanations and misconception types.

# Base descriptions - mapping misconception short names to fuller descriptions
MISCON_DESCRIPTIONS = {
    "NA": "No specific misconception identified",
    "Incomplete": "Incomplete reasoning - the student shows partial understanding but does not fully explain the concept or misses key steps",
    "Additive": "Additive reasoning misconception - the student applies additive logic where multiplicative or proportional reasoning is needed",
    "Duplication": "Duplication error - the student repeats or duplicates numbers or operations incorrectly",
    "Subtraction": "Subtraction error - the student subtracts when a different operation is required, or makes errors in subtraction",
    "Positive": "Positive number bias - the student treats negative numbers as positive or ignores sign rules",
    "Wrong_term": "Wrong term selection - the student identifies or uses the wrong term in a sequence or expression",
    "Irrelevant": "Irrelevant reasoning - the student provides reasoning unrelated to the mathematical concept being tested",
    "Wrong_fraction": "Wrong fraction - the student forms or simplifies fractions incorrectly",
    "Inversion": "Inversion error - the student inverts the fraction, ratio, or operation in the wrong direction",
    "Mult": "Multiplication error - the student makes errors in multiplication or applies multiplication incorrectly",
    "Denominator-only_change": "Denominator-only change - the student only changes the denominator when both parts should change",
    "Whole_numbers_larger": "Whole numbers are larger misconception - the student believes longer decimals or more digits means larger value",
    "Adding_across": "Adding across - the student adds numerators and denominators separately when adding fractions",
    "WNB": "Wrong number bond - the student uses incorrect number bonds or fact relationships",
    "Tacking": "Tacking - the student appends digits without understanding place value in multiplication or addition",
    "Unknowable": "Unknowable - the student claims the answer cannot be determined when it actually can",
    "SwapDividend": "Swap dividend and divisor - the student swaps the dividend and divisor in division",
    "Scale": "Scale misconception - the student misunderstands scale factors or proportional scaling",
    "Not_variable": "Not treating as variable - the student treats a variable as a specific number rather than a general value",
    "Wrong_Fraction": "Wrong Fraction - the student constructs the wrong fraction from the given information",
    "Adding_terms": "Adding terms incorrectly - the student adds algebraic terms that should not be combined",
    "Firstterm": "First term error - the student confuses the first term with the common difference or another element",
    "Multiplying_by_4": "Multiplying by 4 error - the student multiplies by 4 when a different operation or factor is needed",
    "FlipChange": "Flip change error - the student incorrectly flips the sign or direction of change",
    "Shorter_is_bigger": "Shorter is bigger misconception - the student believes shorter decimals represent larger values",
    "CoeffIsExp": "Coefficient as exponent - the student confuses coefficients with exponents",
    "Negative": "Negative sign error - the student makes errors handling negative numbers or signs",
    "Expand_single_bracket": "Expand single bracket error - the student makes errors expanding a single bracket expression",
    "Num-only_change": "Numerator-only change - the student only changes the numerator when both parts should change",
    "Smaller_is_bigger": "Smaller is bigger - the student confuses magnitude ordering of decimal numbers",
}

CATEGORY_DESCRIPTIONS = {
    "True_Correct": "Student answered correctly with proper reasoning",
    "True_Neither": "Student answered correctly but explanation is unclear or irrelevant",
    "True_Misconception": "Student answered correctly but reasoning contains a misconception",
    "False_Correct": "Student answered incorrectly but explanation shows correct reasoning process",
    "False_Neither": "Student answered incorrectly with unclear or irrelevant explanation",
    "False_Misconception": "Student answered incorrectly due to a specific misconception",
}

def build_label_description(label_str: str) -> str:
    """Create a rich text description for each label, used as retrieval corpus."""
    parts = label_str.split(":", 1)
    cat = parts[0]
    miscon = parts[1] if len(parts) > 1 else "NA"
    cat_desc = CATEGORY_DESCRIPTIONS.get(cat, f"Category: {cat}")
    miscon_desc = MISCON_DESCRIPTIONS.get(miscon, f"Misconception: {miscon}")
    if miscon == "NA":
        return f"{cat_desc}. {miscon_desc}."
    else:
        return f"{cat_desc}. The student shows: {miscon_desc}."

# Build descriptions for all 65 labels
LABEL_DESCS = {lab: build_label_description(lab) for lab in classes}
LABEL_DESC_LIST = [LABEL_DESCS[c] for c in classes]  # ordered by class index

print(f"\nTotal classes: {num_labels}")
print(f"True_* classes: {true_mask.sum()}, False_* classes: {false_mask.sum()}")
print(f"\n--- Sample Label Descriptions ---")
for i, (lab, desc) in enumerate(LABEL_DESCS.items()):
    if i < 5:
        print(f"  {lab}: {desc[:100]}...")

# Show sample input text
train_text_full = make_text(train_df, train_df["is_correct"].values.astype(np.int8))
print(f"\n--- Sample Input Text ---")
print(train_text_full[0][:400])


## Cell 5 — Training Components

Advanced training components used by top competition solutions:
- **Focal Loss** (Lin et al., 2017): Down-weights easy examples, focuses training on hard/rare classes
- **AWP** (Wu et al., 2020): Adversarial perturbation for flatter minima
- **R-Drop** (Wu et al., 2021): Consistency regularization via dual forward passes
- **EMA** (Polyak & Juditsky, 1992): Smoother weight averaging
- **Layerwise LR Decay** (Clark et al., 2020): Lower LR for earlier pretrained layers


In [ ]:
# ============================================================
# TRAINING COMPONENTS
# ============================================================
from scipy.special import softmax as scipy_softmax

# --- 5a. Focal Loss (Lin et al., 2017) ---
class FocalLoss(nn.Module):
    """
    FL(p_t) = -α_t × (1 - p_t)^γ × log(p_t)
    Down-weights easy examples, focuses on hard/rare classes.
    Reference: Lin et al., ICCV 2017, arXiv:1708.02002
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_weight = (1.0 - p_t) ** self.gamma
        loss = focal_weight * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss = alpha_t * loss
        if self.reduction == 'mean':
            return loss.mean()
        return loss.sum()


# --- 5a2. Label Smoothing CE (fallback if Focal Loss disabled) ---
class LabelSmoothingCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        log_preds = F.log_softmax(logits, dim=-1)
        nll = F.nll_loss(log_preds, targets, reduction='mean')
        smooth = -log_preds.mean(dim=-1).mean()
        return (1 - self.eps) * nll + self.eps * smooth


# --- 5b. Adversarial Weight Perturbation (AWP) ---
# Reference: Wu et al., NeurIPS 2020, arXiv:2004.05884
class AWP:
    def __init__(self, model, adv_lr=0.1, adv_eps=1e-4):
        self.model = model
        self.adv_lr = adv_lr
        self.adv_eps = adv_eps
        self.backup = {}
        self.backup_eps = {}

    def attack_step(self):
        e = 1e-6
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                norm1 = torch.norm(param.grad)
                norm2 = torch.norm(param.data.detach())
                if norm1 != 0 and not torch.isnan(norm1):
                    r_at = self.adv_lr * param.grad / (norm1 + e) * (norm2 + e)
                    param.data.add_(r_at)
                    param.data = torch.min(
                        torch.max(param.data, self.backup_eps[name][0]),
                        self.backup_eps[name][1]
                    )

    def save(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                self.backup[name] = param.data.clone()
                grad_eps = self.adv_eps * param.abs().detach()
                self.backup_eps[name] = (
                    self.backup[name] - grad_eps,
                    self.backup[name] + grad_eps,
                )

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}
        self.backup_eps = {}


# --- 5c. Exponential Moving Average (EMA) ---
# Reference: Polyak & Juditsky, 1992
class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_avg = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_avg.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}


# --- 5d. R-Drop KL Divergence Loss ---
# Reference: Wu et al., NeurIPS 2021, arXiv:2106.14448
def compute_kl_loss(p, q):
    p_loss = F.kl_div(F.log_softmax(p, dim=-1), F.softmax(q, dim=-1), reduction='batchmean')
    q_loss = F.kl_div(F.log_softmax(q, dim=-1), F.softmax(p, dim=-1), reduction='batchmean')
    return (p_loss + q_loss) / 2


# --- 5e. Layerwise Learning Rate Decay ---
# Reference: Clark et al., ICLR 2020, arXiv:2003.10555
def get_layerwise_lr_params(model, lr, wd, decay_rate):
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    layers = [model.deberta.embeddings] + list(model.deberta.encoder.layer)
    layers.reverse()
    groups = []
    for i, layer in enumerate(layers):
        layer_lr = lr * (decay_rate ** i)
        groups += [
            {"params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
             "weight_decay": wd, "lr": layer_lr},
            {"params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
             "weight_decay": 0.0, "lr": layer_lr},
        ]
    groups += [
        {"params": [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": wd, "lr": lr},
        {"params": [p for n, p in model.classifier.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay": 0.0, "lr": lr},
    ]
    return groups


# --- 5f. Learned Temperature Scaling ---
# Reference: Guo et al., ICML 2017, arXiv:1706.04599
def learn_temperature(val_logits, val_labels, lr=0.01, max_iter=200):
    T_param = nn.Parameter(torch.ones(1) * 1.5)
    logits_t = torch.tensor(val_logits, dtype=torch.float32)
    labels_t = torch.tensor(val_labels, dtype=torch.long)
    opt = torch.optim.LBFGS([T_param], lr=lr, max_iter=max_iter)
    def closure():
        opt.zero_grad()
        scaled = logits_t / T_param.clamp(min=0.1)
        loss = F.cross_entropy(scaled, labels_t)
        loss.backward()
        return loss
    opt.step(closure)
    learned_T = T_param.item()
    print(f"  Learned temperature: {learned_T:.4f}")
    return max(learned_T, 0.1)


# --- 5g. Post-processing helpers ---
def apply_filters(logits, is_correct, temperature=None):
    out = logits.copy()
    T = temperature if temperature is not None else TEMPERATURE
    if USE_TEMPERATURE_SCALING:
        out = out / T
    if USE_FAMILY_FILTER:
        big_neg = -1e9
        for i, ic in enumerate(is_correct):
            if ic == 1:
                out[i, ~true_mask] = big_neg
            elif ic == 0:
                out[i, ~false_mask] = big_neg
    return out


def top3_from_logits(logits):
    idx = np.argsort(-logits, axis=1)[:, :10]
    out = []
    for row in idx:
        seen, picks = set(), []
        for j in row:
            if int(j) not in seen:
                seen.add(int(j))
                picks.append(int(j))
            if len(picks) == 3:
                break
        while len(picks) < 3:
            for j in row:
                if int(j) not in seen:
                    picks.append(int(j))
                    seen.add(int(j))
                if len(picks) == 3:
                    break
            break
        out.append(picks)
    return out


def map_at_3(y_true, y_pred_top3):
    score = 0.0
    for yt, preds in zip(y_true, y_pred_top3):
        for rank, p in enumerate(preds[:3], start=1):
            if p == yt:
                score += 1.0 / rank
                break
    return score / len(y_true)


# FIX: Use "." instead of "/" so it works on both Windows and Linux/Kaggle
def _free_gb(path="."):
    return shutil.disk_usage(path).free / 1e9

print("All training components defined.")
print(f"  Focal Loss: {USE_FOCAL_LOSS} (γ={FOCAL_GAMMA}, class_weights={USE_CLASS_WEIGHTS})")
print(f"  AWP: {USE_AWP} | R-Drop: {USE_RDROP} | EMA: {USE_EMA}")

## Cell 6 — Model A: DeBERTa-v3-large Classifier Training

**Architecture**: `microsoft/deberta-v3-large` → 65-class linear classification head  
**Loss**: Focal Loss (γ=2.0) with inverse-frequency class weights — critical for extreme imbalance (up to 740:1 ratio)  
**Precision**: bf16 mixed precision (H100 native); `GradScaler` is **disabled for bf16**, only enabled for fp16  
**Regularization**: AWP + R-Drop + EMA + Layerwise LR Decay (rate=0.95 per layer)  
**CV**: GroupKFold(5) × 2 seeds × 4 epochs — stays within 12-hr Kaggle limit  

```
For each seed in [42, 123]:
    best_fold_map3 = -1.0   ← per-seed best-checkpoint tracker
    best_fold_path = None

    For each fold in GroupKFold(n_splits=N_FOLDS, groups=QuestionId):
        1. Build correct_by_q from TRAIN fold only (prevents leakage)
        2. Build structured input: "Question: … Answer: … Is Correct: … Explanation: …"
        3. Load DeBERTa-v3-large fresh each fold; configure layerwise LR optimizer
        4. Cosine LR schedule with warmup (WARMUP_RATIO=0.1 of total steps)

        For each epoch (active_epochs = EPOCHS = 4):
            For each gradient-accumulation window (GRAD_ACCUM=2 steps):
                a. bf16 autocast forward pass → logits1
                b. R-Drop: second forward pass → logits2
                   loss = (FocalLoss(logits1) + FocalLoss(logits2)) / 2
                         + RDROP_ALPHA × KL(logits1 ‖ logits2)
                c. scaler.scale(loss / GRAD_ACCUM).backward()
                d. AWP (if epoch >= AWP_START_EPOCH=1):
                   awp.save() → awp.attack_step() → adv forward → adv backward → awp.restore()
                   NOTE: AWP step runs BEFORE scaler.unscale_() so all gradients
                   remain consistently scaled when unscale_() is called once.
                e. scaler.unscale_() → clip_grad_norm_(MAX_GRAD_NORM=1.0)
                   scaler.step() → scaler.update() → scheduler.step()
                f. EMA update (decay=0.999)

        Evaluate with EMA shadow weights → OOF logits stored for ensemble
        Fold MAP@3 computed → if best so far, save checkpoint (with 2 GB disk guard)

After all folds/seeds:
    Learn temperature T via LBFGS on full OOF logits (Guo et al., 2017)
    Final DeBERTa OOF MAP@3 reported
```


### Tokenizer & Data Preparation

Initialize tokenizer and collator once, shared across all folds/seeds.


In [ ]:
# ============================================================
# TOKENIZER + DATA PREPARATION
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL, use_fast=True)
collator  = DataCollatorWithPadding(tokenizer)
print(f"Tokenizer: {CLASSIFIER_MODEL} (vocab={tokenizer.vocab_size})")

# Verify bf16 support
if USE_BF16 and torch.cuda.is_bf16_supported():
    print("bf16 mixed precision: ENABLED (H100 native)")
    AMP_DTYPE = torch.bfloat16
elif torch.cuda.is_available():
    print("bf16 not supported, using fp16 mixed precision")
    AMP_DTYPE = torch.float16
    USE_BF16 = False
else:
    AMP_DTYPE = torch.float32
    USE_BF16 = False

# Set up loss function
if USE_FOCAL_LOSS:
    _alpha = class_weights_tensor.to(device) if USE_CLASS_WEIGHTS else None
    print(f"Loss: FocalLoss(γ={FOCAL_GAMMA}, class_weights={'yes' if USE_CLASS_WEIGHTS else 'no'})")
else:
    _alpha = None
    print(f"Loss: LabelSmoothingCE(ε={LABEL_SMOOTHING})")

print(f"\nReady for training: {len(SEEDS)} seeds × {N_FOLDS} folds × {EPOCHS} epochs")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")


In [ ]:
# ============================================================
# MODEL A: DEBERTA-V3-LARGE TRAINING  (Focal Loss + bf16 + AWP + R-Drop + EMA)
# ============================================================

groups = train_df["QuestionId"].values
gkf    = GroupKFold(n_splits=N_FOLDS)

# Out-of-fold logits storage: [N_samples, N_classes, N_seeds]
all_oof_logits = np.zeros((len(train_df), num_labels, len(SEEDS)), dtype=np.float32)
fold_scores    = []
oof_top3_labels = [None] * len(train_df)

# QUICK_RUN already overrode EPOCHS and N_FOLDS in the config cell — use them directly
active_epochs = EPOCHS
active_folds  = N_FOLDS

# Create model save directory
if SAVE_FOLD_MODELS:
    MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training: {len(SEEDS)} seeds × {active_folds} folds × {active_epochs} epochs")
print(f"Model: {CLASSIFIER_MODEL}")
print(f"Device: {device} | Precision: {'bf16' if USE_BF16 else 'fp16/fp32'}")
print(f"Loss: {'FocalLoss' if USE_FOCAL_LOSS else 'LabelSmoothingCE'}")
print("=" * 70)

for seed_idx, seed in enumerate(SEEDS):
    set_seed(seed)
    print(f"\n{'='*70}")
    print(f"  SEED {seed} ({seed_idx+1}/{len(SEEDS)})")
    print(f"{'='*70}")

    # ── Best-fold tracking (OUTSIDE fold loop to avoid saving every fold) ──
    best_fold_map3 = -1.0
    best_fold_path = None

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(np.zeros(len(train_df)), y_all, groups), start=1):
        if fold > active_folds:
            break

        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        va_df = train_df.iloc[va_idx].reset_index(drop=True)

        # Infer correct answers from TRAIN fold only (no leakage!)
        correct_by_q = build_correct_by_q(tr_df)
        tr_is_correct = compute_is_correct(tr_df, correct_by_q)
        va_is_correct = compute_is_correct(va_df, correct_by_q)

        # Build text inputs
        tr_text = make_text(tr_df, tr_is_correct)
        va_text = make_text(va_df, va_is_correct)
        tr_y = tr_df["label_str"].map(lab2id).astype(int).values
        va_y = va_df["label_str"].map(lab2id).astype(int).values

        # Create HF Datasets
        def tok_fn(batch):
            return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

        tr_ds = Dataset.from_dict({"text": tr_text, "labels": tr_y.tolist()})
        va_ds = Dataset.from_dict({"text": va_text, "labels": va_y.tolist()})
        tr_ds = tr_ds.map(tok_fn, batched=True, remove_columns=["text"]).with_format("torch")
        va_ds = va_ds.map(tok_fn, batched=True, remove_columns=["text"]).with_format("torch")

        # Load fresh model
        model = AutoModelForSequenceClassification.from_pretrained(
            CLASSIFIER_MODEL, num_labels=num_labels,
            ignore_mismatched_sizes=True,
        ).to(device)

        # Optimizer with layerwise LR decay
        if USE_LAYERWISE_LR_DECAY:
            opt_params = get_layerwise_lr_params(model, LR, WEIGHT_DECAY, LAYERWISE_LR_DECAY)
            optimizer = AdamW(opt_params, lr=LR, eps=1e-8)
        else:
            optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-8)

        # Cosine LR schedule with warmup
        num_steps  = int(len(tr_ds) / (BATCH_SIZE * GRAD_ACCUM) * active_epochs)
        num_warmup = int(num_steps * WARMUP_RATIO)
        scheduler  = get_cosine_schedule_with_warmup(optimizer, num_warmup, num_steps)

        # Loss function: Focal Loss (with class weights) or LabelSmoothingCE
        if USE_FOCAL_LOSS:
            criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=_alpha)
        elif LABEL_SMOOTHING > 0:
            criterion = LabelSmoothingCE(LABEL_SMOOTHING)
        else:
            criterion = nn.CrossEntropyLoss()

        # Advanced components
        awp = AWP(model, AWP_LR, AWP_EPS) if USE_AWP else None
        ema = EMA(model, EMA_DECAY)       if USE_EMA else None

        # FIX: GradScaler is only needed for fp16 (bf16 has same dynamic range as fp32)
        scaler = GradScaler("cuda", enabled=USE_FP16)

        # DataLoader
        train_loader = torch.utils.data.DataLoader(
            tr_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator
        )

        print(f"\n--- Fold {fold}/{active_folds} | Train: {len(tr_ds)} | Val: {len(va_ds)} ---")
        print(f"    Steps: {num_steps} | Warmup: {num_warmup}")

        # ===================== TRAINING LOOP (bf16 mixed precision) =====================
        model.train()
        optimizer.zero_grad()

        for epoch in range(active_epochs):
            epoch_loss = 0.0
            for step, batch in enumerate(train_loader):
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop("labels")

                # bf16 autocast for forward + loss
                with autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(AMP_DTYPE != torch.float32)):
                    outputs = model(**batch)
                    logits1 = outputs.logits

                    if USE_RDROP:
                        outputs2 = model(**batch)
                        logits2  = outputs2.logits
                        ce_loss  = (criterion(logits1, labels) + criterion(logits2, labels)) / 2
                        kl_loss  = compute_kl_loss(logits1, logits2)
                        loss     = ce_loss + RDROP_ALPHA * kl_loss
                    else:
                        loss = criterion(logits1, labels)

                    loss = loss / GRAD_ACCUM

                # Scaled backward
                scaler.scale(loss).backward()
                epoch_loss += loss.item() * GRAD_ACCUM

                if (step + 1) % GRAD_ACCUM == 0:
                    # FIX: AWP adversarial step BEFORE unscale so all gradients
                    # are consistently scaled when unscale_ is called once.
                    if USE_AWP and epoch >= AWP_START_EPOCH:
                        awp.save()
                        awp.attack_step()
                        with autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(AMP_DTYPE != torch.float32)):
                            out_adv = model(**batch)
                            loss_adv = criterion(out_adv.logits, labels) / GRAD_ACCUM
                        scaler.scale(loss_adv).backward()
                        awp.restore()

                    # Unscale ALL gradients together, then clip
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

                    if USE_EMA:
                        ema.update()

            avg_loss = epoch_loss / len(train_loader)
            print(f"    Epoch {epoch+1}/{active_epochs} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

        # ===================== EVALUATION =====================
        if USE_EMA:
            ema.apply_shadow()

        model.eval()
        val_loader = torch.utils.data.DataLoader(
            va_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator
        )

        all_logits = []
        with torch.no_grad():
            with autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(AMP_DTYPE != torch.float32)):
                for batch in val_loader:
                    batch = {k: v.to(device) for k, v in batch.items()}
                    batch.pop("labels", None)
                    outputs = model(**batch)
                    all_logits.append(outputs.logits.float().cpu().numpy())

        va_logits = np.concatenate(all_logits, axis=0)

        # Store OOF logits (EMA weights' predictions)
        for ii, ridx in enumerate(va_idx):
            all_oof_logits[ridx, :, seed_idx] = va_logits[ii]

        # Evaluate fold
        va_logits_filtered = apply_filters(va_logits, va_is_correct)
        va_top3 = top3_from_logits(va_logits_filtered)
        fold_map3 = map_at_3(va_y, va_top3)
        fold_scores.append(fold_map3)
        print(f"    Fold MAP@3: {fold_map3:.6f}")

        # Save best model per seed (with disk guard)
        if SAVE_FOLD_MODELS and fold_map3 > best_fold_map3:
            free = _free_gb()
            if free < 2.0:
                print(f"    ⚠ Only {free:.1f} GB free — skipping save!")
            else:
                if best_fold_path is not None and best_fold_path.exists():
                    shutil.rmtree(best_fold_path)
                    print(f"    Removed old checkpoint: {best_fold_path.name}")
                best_fold_map3 = fold_map3
                best_fold_path = MODEL_SAVE_DIR / f"seed{seed}_fold{fold}"
                model.save_pretrained(best_fold_path)
                tokenizer.save_pretrained(best_fold_path)
                print(f"    Saved best → {best_fold_path.name}  ({free:.1f} GB free)")
        else:
            best_fold_map3 = max(best_fold_map3, fold_map3)

        if USE_EMA:
            ema.restore()

        for ii, ridx in enumerate(va_idx):
            oof_top3_labels[ridx] = [id2lab[j] for j in va_top3[ii]]

        # Cleanup fold — aggressive memory release
        del model, optimizer, scheduler, awp, ema, scaler
        del train_loader, val_loader, tr_ds, va_ds, all_logits, va_logits
        torch.cuda.empty_cache()
        gc.collect()

# ============================================================
# LEARN TEMPERATURE FROM OOF LOGITS (Guo et al., 2017)
# ============================================================
print(f"\n{'='*70}")
print("  TEMPERATURE LEARNING + OOF EVALUATION")
print(f"{'='*70}")

deberta_oof_logits = all_oof_logits.mean(axis=2)  # [N, C] averaged across seeds
LEARNED_TEMPERATURE = learn_temperature(deberta_oof_logits, y_all)

# DeBERTa-only ensemble score
is_correct_full = train_df["is_correct"].values
deberta_filtered = apply_filters(deberta_oof_logits, is_correct_full, temperature=LEARNED_TEMPERATURE)
deberta_top3 = top3_from_logits(deberta_filtered)
deberta_map3 = map_at_3(y_all, deberta_top3)

print(f"\nModel A (DeBERTa) Results:")
print(f"  Individual fold MAP@3: {[round(s, 5) for s in fold_scores]}")
print(f"  Mean fold MAP@3:       {np.mean(fold_scores):.6f}")
print(f"  DeBERTa OOF MAP@3:     {deberta_map3:.6f}")
print(f"  Learned temperature:   {LEARNED_TEMPERATURE:.4f}")
print(f"{'='*70}")

## Cell 7 — Model B: Sentence Transformer (BGE-Large) Fine-Tuning

**Architecture**: `BAAI/bge-large-en-v1.5` bi-encoder → cosine similarity retrieval  
**Loss**: `MultipleNegativesRankingLoss` — treats all other samples in the batch as negatives (**in-batch negatives only**, batch size = 48 → ~47 negatives per positive)  
**Goal**: Learn embeddings where student text is close to the correct label description  
**CV**: GroupKFold(5) × 3 epochs; only the **best-performing fold** checkpoint is saved to `retriever_model/` (not the last fold)  

**Retrieval scoring** (`retrieval_probs`):
1. Encode query texts and all 65 label descriptions with `normalize_embeddings=True`
2. Cosine similarity matrix: `sim = text_embs @ label_embs.T` → shape `[N, 65]`
3. Scale: `sim × 20.0` (temperature sharpening to logit-like range)
4. `scipy.special.softmax(sim_matrix, axis=1)` — numerically stable, avoids `np.exp` overflow

The retrieval model provides **diversity** to the ensemble — it captures semantic similarity between misconceptions that the classifier may miss. For example, two confusable labels with similar descriptions will be scored similarly by retrieval, allowing the ensemble to rank the correct one higher.

**References**:
- Reimers & Gurevych (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP 2019. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
- Xiao et al. (2024). *C-Pack: Packaged Resources for General Chinese Embeddings.* SIGIR 2024. [arXiv:2309.07597](https://arxiv.org/abs/2309.07597)
- Henderson et al. (2017). *Efficient Natural Language Response Suggestion for Smart Reply.* [arXiv:1705.00652](https://arxiv.org/abs/1705.00652) — MultipleNegativesRankingLoss


In [ ]:
# ============================================================
# MODEL B: SENTENCE TRANSFORMER FINE-TUNING (BGE-Large)
# ============================================================
# Uses MultipleNegativesRankingLoss to learn embeddings where
# student text → correct label description (cosine similarity).
# Then at inference: embed test text, compute cosine sim vs all
# 65 label descriptions → top-3 retrieval.

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader as STDataLoader

# Ensure VRAM is clear before loading retriever
torch.cuda.empty_cache()
gc.collect()

# ── Build training pairs: (student_text, label_description) ──
def build_st_examples(df, is_correct_arr):
    """Create InputExample pairs for sentence transformer training."""
    examples = []
    texts = make_text(df, is_correct_arr)
    for i, row in df.iterrows():
        label = row["label_str"]
        if label in LABEL_DESCS:
            examples.append(InputExample(
                texts=[texts[i], LABEL_DESCS[label]]
            ))
    return examples

# ── Retrieval scoring function ──
def retrieval_probs(model, texts, label_descs_list):
    """
    Compute retrieval probabilities: cosine_sim(text_emb, label_emb) → softmax → probs
    Returns: [N, num_labels] probability matrix
    """
    text_embs = model.encode(texts, batch_size=RETRIEVER_BATCH, show_progress_bar=False, normalize_embeddings=True)
    label_embs = model.encode(label_descs_list, batch_size=RETRIEVER_BATCH, show_progress_bar=False, normalize_embeddings=True)
    # Cosine similarity (already normalized)
    sim_matrix = text_embs @ label_embs.T  # [N, num_labels]
    # Scale to logit-like range and use numerically stable softmax
    sim_matrix = sim_matrix * 20.0  # temperature scaling for sharper distribution
    # FIX: Use scipy_softmax instead of manual exp — avoids overflow for large values
    probs = scipy_softmax(sim_matrix, axis=1)
    return probs

# ── Label descriptions in correct order ──
label_keys_ordered = [id2lab[i] for i in range(num_labels)]
label_descs_ordered = [LABEL_DESCS.get(k, k) for k in label_keys_ordered]

# ── OOF retrieval storage ──
retrieval_oof_probs = np.zeros((len(train_df), num_labels), dtype=np.float32)

print(f"Sentence Transformer: {RETRIEVER_MODEL}")
print(f"Epochs: {RETRIEVER_EPOCHS} | Batch: {RETRIEVER_BATCH}")
print(f"Label descriptions: {len(label_descs_ordered)}")
print("=" * 70)

gkf_st = GroupKFold(n_splits=N_FOLDS)

# FIX: Track best retriever fold and save that one, not just last fold
best_ret_map3 = -1.0

for fold, (tr_idx, va_idx) in enumerate(gkf_st.split(np.zeros(len(train_df)), y_all, groups), start=1):
    if fold > active_folds:
        break

    tr_df_st = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df_st = train_df.iloc[va_idx].reset_index(drop=True)

    correct_by_q_st = build_correct_by_q(tr_df_st)
    tr_is_correct_st = compute_is_correct(tr_df_st, correct_by_q_st)
    va_is_correct_st = compute_is_correct(va_df_st, correct_by_q_st)

    # Build training examples
    train_examples = build_st_examples(tr_df_st, tr_is_correct_st)
    print(f"\n--- ST Fold {fold}/{active_folds} | Examples: {len(train_examples)} ---")

    # Load fresh model each fold
    st_model = SentenceTransformer(RETRIEVER_MODEL, device=device)

    # DataLoader and loss
    st_train_dl = STDataLoader(train_examples, shuffle=True, batch_size=RETRIEVER_BATCH)
    st_loss = losses.MultipleNegativesRankingLoss(st_model)

    # Train
    warmup_steps = int(len(st_train_dl) * RETRIEVER_EPOCHS * 0.1)
    st_model.fit(
        train_objectives=[(st_train_dl, st_loss)],
        epochs=RETRIEVER_EPOCHS,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
        optimizer_params={"lr": 2e-5},
    )

    # OOF retrieval scoring
    va_texts = make_text(va_df_st, va_is_correct_st)
    va_probs = retrieval_probs(st_model, va_texts, label_descs_ordered)

    for ii, ridx in enumerate(va_idx):
        retrieval_oof_probs[ridx] = va_probs[ii]

    # Evaluate retrieval fold
    va_y_st = va_df_st["label_str"].map(lab2id).astype(int).values
    ret_top3 = top3_from_logits(va_probs)
    ret_map3 = map_at_3(va_y_st, ret_top3)
    print(f"    Retrieval Fold MAP@3: {ret_map3:.6f}")

    # FIX: Save best-performing retriever fold (not just last)
    if ret_map3 > best_ret_map3:
        best_ret_map3 = ret_map3
        st_model.save("retriever_model")
        print(f"    Saved best retriever model (MAP@3={ret_map3:.6f})")

    del st_model, st_train_dl, train_examples
    torch.cuda.empty_cache()
    gc.collect()

# Overall retrieval OOF score
ret_oof_top3 = top3_from_logits(retrieval_oof_probs)
retrieval_oof_map3 = map_at_3(y_all, ret_oof_top3)
print(f"\nModel B (Retrieval) OOF MAP@3: {retrieval_oof_map3:.6f}")
print("=" * 70)

## Cell 8 — Two-Model Ensemble + OOF Evaluation

**Ensemble Strategy**: α × DeBERTa_probs + (1-α) × Retrieval_probs  
**α Selection**: Grid search on OOF data (α ∈ [0.0, 0.05, ..., 1.0])  

The ensemble works because:
1. **DeBERTa** excels at discriminative classification — high precision on common classes
2. **Retrieval** captures semantic similarity — better on rare/confusable classes
3. Their errors are partially uncorrelated → ensemble reduces both bias and variance

> **Reference**: Lakshminarayanan et al. (2017). *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles.* NeurIPS 2017. [arXiv:1612.01474](https://arxiv.org/abs/1612.01474)


In [ ]:
# ============================================================
# TWO-MODEL ENSEMBLE + OOF EVALUATION
# ============================================================

# Convert DeBERTa logits to probabilities (with learned temperature)
from scipy.special import softmax as scipy_softmax

deberta_probs = scipy_softmax(deberta_oof_logits / LEARNED_TEMPERATURE, axis=1)

# ── Grid search for optimal ensemble weight α ──
print("Searching for optimal ensemble weight α...")
best_alpha = 0.5
best_ensemble_map3 = 0.0
alpha_results = []

for alpha_candidate in np.arange(0.0, 1.05, 0.05):
    alpha_candidate = round(alpha_candidate, 2)
    combined_probs = alpha_candidate * deberta_probs + (1 - alpha_candidate) * retrieval_oof_probs

    # Apply family filter on combined probs
    combined_filtered = combined_probs.copy()
    if USE_FAMILY_FILTER:
        big_neg = 0.0
        for i, ic in enumerate(is_correct_full):
            if ic == 1:
                combined_filtered[i, ~true_mask] = big_neg
            elif ic == 0:
                combined_filtered[i, ~false_mask] = big_neg

    cand_top3 = top3_from_logits(combined_filtered)
    cand_map3 = map_at_3(y_all, cand_top3)
    alpha_results.append((alpha_candidate, cand_map3))

    if cand_map3 > best_ensemble_map3:
        best_ensemble_map3 = cand_map3
        best_alpha = alpha_candidate

ENSEMBLE_ALPHA = best_alpha

print(f"\nAlpha search results:")
for a, m in alpha_results:
    marker = " ← BEST" if a == best_alpha else ""
    print(f"  α={a:.2f}: MAP@3={m:.6f}{marker}")

# ── Final ensemble with optimal α ──
final_probs = ENSEMBLE_ALPHA * deberta_probs + (1 - ENSEMBLE_ALPHA) * retrieval_oof_probs
final_filtered = final_probs.copy()
if USE_FAMILY_FILTER:
    for i, ic in enumerate(is_correct_full):
        if ic == 1:
            final_filtered[i, ~true_mask] = 0.0
        elif ic == 0:
            final_filtered[i, ~false_mask] = 0.0

ensemble_top3 = top3_from_logits(final_filtered)
ensemble_map3 = map_at_3(y_all, ensemble_top3)

for i in range(len(train_df)):
    oof_top3_labels[i] = [id2lab[j] for j in ensemble_top3[i]]

train_df["pred_top3"] = [" ".join(p) for p in oof_top3_labels]

print(f"\n{'='*70}")
print(f"  FINAL ENSEMBLE RESULTS")
print(f"{'='*70}")
print(f"  Model A (DeBERTa) OOF MAP@3:    {deberta_map3:.6f}")
print(f"  Model B (Retrieval) OOF MAP@3:   {retrieval_oof_map3:.6f}")
print(f"  Ensemble (α={ENSEMBLE_ALPHA:.2f}) MAP@3:     {ensemble_map3:.6f}")
print(f"  Improvement over DeBERTa alone:  {(ensemble_map3 - deberta_map3)*100:.2f}%")
print(f"{'='*70}")

# ── Evaluation Plots ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Alpha vs MAP@3
ax = axes[0]
alphas = [a for a, _ in alpha_results]
maps = [m for _, m in alpha_results]
ax.plot(alphas, maps, 'b-o', linewidth=2, markersize=6)
ax.axvline(ENSEMBLE_ALPHA, color='red', linestyle='--', label=f'Best α={ENSEMBLE_ALPHA:.2f}')
ax.set_xlabel("α (DeBERTa weight)", fontsize=12)
ax.set_ylabel("MAP@3", fontsize=12)
ax.set_title("Ensemble Weight Search", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Model comparison
ax = axes[1]
model_names = ["DeBERTa\n(classifier)", "BGE-Large\n(retrieval)", "Ensemble"]
model_scores = [deberta_map3, retrieval_oof_map3, ensemble_map3]
colors = ["#3498db", "#e67e22", "#2ecc71"]
bars = ax.bar(model_names, model_scores, color=colors, edgecolor="black", alpha=0.85)
for bar, score in zip(bars, model_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{score:.4f}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("MAP@3", fontsize=12)
ax.set_title("Model Comparison", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.0)

# 3. Per-category accuracy
pred_top1 = [p[0] if p else "?" for p in oof_top3_labels]
train_df["pred_top1"] = pred_top1
train_df["pred_cat"] = train_df["pred_top1"].str.split(":").str[0]
train_df["true_cat"] = train_df["Category"]
cat_accuracy = {}
for cat in train_df["true_cat"].unique():
    mask = train_df["true_cat"] == cat
    if mask.sum() > 0:
        cat_accuracy[cat] = (train_df.loc[mask, "pred_cat"] == cat).mean()
ax = axes[2]
cats = sorted(cat_accuracy.keys())
accs = [cat_accuracy[c] for c in cats]
ax.barh(cats, accs, color=sns.color_palette("Set2", len(cats)), edgecolor="black")
ax.set_title("Category Accuracy", fontsize=14, fontweight="bold")
ax.set_xlabel("Accuracy")
ax.set_xlim(0, 1.0)
for i, (c, a) in enumerate(zip(cats, accs)):
    ax.text(a + 0.01, i, f"{a:.2%}", va="center", fontsize=10)

plt.tight_layout()
plt.suptitle("Two-Model Ensemble Performance", fontsize=16, fontweight="bold", y=1.03)
plt.show()

# Print OOF summary
top1_acc = sum(1 for i in range(len(train_df)) if oof_top3_labels[i] and oof_top3_labels[i][0] == train_df.iloc[i]["label_str"]) / len(train_df)
top3_acc = sum(1 for i in range(len(train_df)) if oof_top3_labels[i] and train_df.iloc[i]["label_str"] in oof_top3_labels[i][:3]) / len(train_df)
print(f"\n  OOF Top-1 Accuracy: {top1_acc:.4f}")
print(f"  OOF Top-3 Accuracy: {top3_acc:.4f}")


## Cell 9 — Explainable AI: Knowledge Base + Teacher Feedback

Builds a **Misconception Knowledge Base (KB)** using LLM-generated structured cards, then generates **personalized teacher feedback** combining model predictions + KB entries. Provides real-world pedagogical value beyond raw MAP@3 predictions.

**LLM Stack**:
- **Primary**: `Qwen/Qwen2.5-7B-Instruct` loaded in **4-bit NF4 quantization** via BitsAndBytes  
  (`load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_compute_dtype=float16`, `bnb_4bit_use_double_quant=True`) — ~4 GB VRAM
- **Fallback**: `google/flan-t5-large` (seq2seq, full precision, ~3 GB VRAM) if 4-bit GPU load fails
- **Skipped entirely** when `QUICK_RUN=True` or `USE_LLM_EXPLAINER=False`

**KB Card Structure** (JSON per misconception type, cached to `misconception_kb.json`):

| Field | Content |
|-------|---------|
| `plain_definition` | Plain-language explanation of the misconception |
| `typical_wrong_thought` | Common student reasoning error pattern |
| `correct_rule` | What the correct mathematical rule actually is |
| `fix_steps` | Step-by-step correction guide (list) |
| `quick_check` | Self-verification question for the student |
| `tiny_example` | Small worked example |

**Teacher Feedback** (`generate_teacher_feedback`): for each prediction, combines the student's answer, the correct answer, the predicted misconception label, the KB plain definition, fix steps, and a self-check question into a structured feedback message.

**References**:
- Qwen Team (2024). *Qwen2.5 Technical Report.* [arXiv:2412.15115](https://arxiv.org/abs/2412.15115)
- Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* NeurIPS 2023. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)


In [ ]:
# ============================================================
# EXPLAINABLE AI — KNOWLEDGE BASE + TEACHER FEEDBACK
# ============================================================

# --- KB Builder ---
def load_explainer():
    if not USE_LLM_EXPLAINER:
        return None, None, None
    from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM
    if device == "cuda":
        try:
            from transformers import BitsAndBytesConfig
            bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                         bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
            tok = AutoTokenizer.from_pretrained(EXPLAINER_MODEL, use_fast=True)
            mdl = AutoModelForCausalLM.from_pretrained(EXPLAINER_MODEL, quantization_config=bnb_cfg,
                                                        device_map="auto", torch_dtype=torch.float16)
            mdl.eval()
            print(f"Loaded explainer (4-bit GPU): {EXPLAINER_MODEL}")
            return "causal", mdl, tok
        except Exception as e:
            print(f"4-bit explainer failed: {str(e)[:100]}")
    tok = AutoTokenizer.from_pretrained(EXPLAINER_FALLBACK_MODEL, use_fast=True)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(EXPLAINER_FALLBACK_MODEL).to(device)
    mdl.eval()
    print(f"Loaded explainer (fallback): {EXPLAINER_FALLBACK_MODEL}")
    return "seq2seq", mdl, tok

def llm_generate(prompt, max_new_tokens=320, ek=None, em=None, et=None):
    if em is None:
        return ""
    if ek == "causal":
        inputs = et(prompt, return_tensors="pt", truncation=True, max_length=2048).to(em.device)
        with torch.no_grad():
            out = em.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1)
        text = et.decode(out[0], skip_special_tokens=True)
        return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()
    else:
        inputs = et(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            out = em.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
        return et.decode(out[0], skip_special_tokens=True).strip()

EXPLAINER_KIND, explainer_model, explainer_tok = load_explainer()

# Build KB
train_df["miscon_only"] = train_df["Misconception"].astype(str)
freq = Counter(train_df["miscon_only"].tolist())
top_miscons = [m for m, _ in freq.most_common(KB_BUILD_TOP_N) if m != "NA"]
print(f"Building KB for {len(top_miscons)} misconception types...")

if KB_PATH.exists():
    kb = json.loads(KB_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cached KB: {len(kb)} entries")
else:
    kb = {}
    for i, miscon in enumerate(top_miscons, start=1):
        sub = train_df[train_df["miscon_only"] == miscon]
        sample = sub.sample(n=min(KB_EXAMPLES_PER_MISCON, len(sub)), random_state=42)
        examples = [{"question": str(r["QuestionText"])[:200], "student_answer": str(r["MC_Answer"]),
                      "student_expl": str(r["StudentExplanation"])[:200]} for _, r in sample.iterrows()]
        prompt = f"You are a math teacher. Misconception: {miscon}\nExamples: {json.dumps(examples)}\nReturn JSON with keys: plain_definition, typical_wrong_thought, correct_rule, fix_steps, quick_check, tiny_example."
        out = llm_generate(prompt, 420, EXPLAINER_KIND, explainer_model, explainer_tok)
        card = None
        if out:
            m = re.search(r"\{.*\}", out, flags=re.S)
            if m:
                try: card = json.loads(m.group(0))
                except: card = None
        if card is None:
            card = {"plain_definition": f"Students make this mistake: {miscon}.",
                    "fix_steps": ["Restate goal.", "Write correct rule.", "Apply step-by-step.", "Check with easy example."],
                    "quick_check": "Can you test your rule on a simpler number?"}
        kb[miscon] = card
        if i % 20 == 0: print(f"  Built {i}/{len(top_miscons)} cards...")
    KB_PATH.write_text(json.dumps(kb, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved KB: {len(kb)} entries")

# --- Teacher Feedback Generator ---
def generate_teacher_feedback(row, pred_label, pred_top3):
    miscon = pred_label.split(":", 1)[1] if ":" in pred_label else pred_label
    card = kb.get(miscon, {})
    ic = int(row.get("is_correct", -1))
    correct = str(correct_by_q_full.get(row["QuestionId"], "?"))
    steps = card.get("fix_steps", ["Re-read the question.", "Write the correct rule.", "Apply step-by-step.", "Check with simpler numbers."])
    lines = [f"Student chose: {row['MC_Answer']} | Correct: {correct}",
             f"Predicted misconception: {pred_label}"]
    if "plain_definition" in card:
        lines.append(f"Definition: {card['plain_definition']}")
    lines.append("\nFix steps:")
    for k, s in enumerate(steps[:5], 1):
        lines.append(f"  {k}. {s}")
    lines.append(f"\nSelf-check: {card.get('quick_check', 'Does your rule work if numbers change?')}")
    return "\n".join(lines)

# Demo
print("\n" + "=" * 70)
print("  SAMPLE TEACHER FEEDBACK (3 examples)")
print("=" * 70)
for idx in np.random.default_rng(42).choice(len(train_df), size=3, replace=False):
    row = train_df.iloc[idx]
    top3 = row.get("pred_top3", "").split()[:3]
    pred1 = top3[0] if top3 else "Unknown:Unknown"
    print(f"\n{'─'*60}")
    print(f"True: {row['label_str']} | Pred: {top3}")
    print(generate_teacher_feedback(row, pred1, top3))
print("─" * 60)

# Cleanup explainer
if explainer_model is not None:
    del explainer_model, explainer_tok
    torch.cuda.empty_cache()
    gc.collect()
    print("\nExplainer model freed from VRAM.")


## Cell 10 — Test Inference & Submission (Two-Model Ensemble)

Generates predictions using the two-model ensemble in four steps:

1. **DeBERTa inference**: Load saved best-fold checkpoints from `fold_models/seed*_fold*` and average logits across all saved models. **Fallback**: if no checkpoints exist (e.g., `QUICK_RUN=True` sets `SAVE_FOLD_MODELS=False`), retrain DeBERTa on full training data with EMA (no cross-validation).
2. **Retrieval inference**: Load best-fold BGE-Large from `retriever_model/` (fallback: pretrained weights) → encode test texts + all 65 label descriptions → cosine similarity → `scipy_softmax(sim × 20.0, axis=1)`.
3. **Ensemble**: `ENSEMBLE_ALPHA × DeBERTa_probs + (1 - ENSEMBLE_ALPHA) × Retrieval_probs` using OOF-learned `ENSEMBLE_ALPHA`.
4. **Post-process & submit**: Apply family filter (True_*/False_* constraint from `is_correct`) → `top3_from_logits` → `submission.csv`.


In [ ]:
# ============================================================
# TEST INFERENCE & SUBMISSION (TWO-MODEL ENSEMBLE)
# ============================================================

# Ensure VRAM is clear before inference
torch.cuda.empty_cache()
gc.collect()

test_df = pd.read_csv(TEST_PATH)
print(f"Test set: {len(test_df)} samples")

test_df["StudentExplanation"] = test_df["StudentExplanation"].fillna("")
test_df["QuestionText"]       = test_df["QuestionText"].fillna("")
test_df["MC_Answer"]          = test_df["MC_Answer"].fillna("")

# Infer is_correct from full training data
test_is_correct = compute_is_correct(test_df, correct_by_q_full)
test_text = make_text(test_df, test_is_correct)

# ============================================================
# PART 1: DeBERTa Inference
# ============================================================

def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

test_ds = Dataset.from_dict({"text": test_text})
test_ds = test_ds.map(tok_fn, batched=True, remove_columns=["text"]).with_format("torch")
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator
)

saved_models = sorted(MODEL_SAVE_DIR.glob("seed*_fold*")) if MODEL_SAVE_DIR.exists() else []

if len(saved_models) > 0:
    print(f"\nLoading {len(saved_models)} DeBERTa checkpoints for ensemble...")
    test_logits_all = []
    for model_path in saved_models:
        print(f"  {model_path.name}...")
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, num_labels=num_labels, ignore_mismatched_sizes=True
        ).to(device)
        model.eval()
        fold_logits = []
        with torch.no_grad():
            with autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(AMP_DTYPE != torch.float32)):
                for batch in test_loader:
                    batch = {k: v.to(device) for k, v in batch.items()}
                    outputs = model(**batch)
                    fold_logits.append(outputs.logits.float().cpu().numpy())
        test_logits_all.append(np.concatenate(fold_logits, axis=0))
        del model; torch.cuda.empty_cache(); gc.collect()
    deberta_test_logits = np.mean(test_logits_all, axis=0)
    del test_logits_all; gc.collect()
    print(f"  DeBERTa ensemble of {len(saved_models)} models done.")
else:
    print("\nNo saved DeBERTa checkpoints — retraining on full data...")
    full_text = make_text(train_df, train_df["is_correct"].values.astype(np.int8))
    full_ds = Dataset.from_dict({"text": full_text, "labels": y_all.tolist()})
    full_ds = full_ds.map(tok_fn, batched=True, remove_columns=["text"]).with_format("torch")
    set_seed(42)
    model = AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER_MODEL, num_labels=num_labels, ignore_mismatched_sizes=True).to(device)
    if USE_LAYERWISE_LR_DECAY:
        opt_params = get_layerwise_lr_params(model, LR, WEIGHT_DECAY, LAYERWISE_LR_DECAY)
        optimizer = AdamW(opt_params, lr=LR, eps=1e-8)
    else:
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    num_steps = int(len(full_ds) / (BATCH_SIZE * GRAD_ACCUM) * EPOCHS)
    num_warmup = int(num_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup, num_steps)
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=_alpha)
    else:
        criterion = nn.CrossEntropyLoss()
    ema_full = EMA(model, EMA_DECAY) if USE_EMA else None
    # FIX: GradScaler only for fp16 (bf16 doesn't need it)
    scaler = GradScaler("cuda", enabled=USE_FP16)
    train_loader = torch.utils.data.DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
    model.train(); optimizer.zero_grad()
    for epoch in range(EPOCHS):
        epoch_loss = 0
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            with autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(AMP_DTYPE != torch.float32)):
                outputs = model(**batch)
                loss = criterion(outputs.logits, labels) / GRAD_ACCUM
            scaler.scale(loss).backward(); epoch_loss += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad()
                if ema_full: ema_full.update()
        print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss/len(train_loader):.4f}")
    if ema_full: ema_full.apply_shadow()
    model.eval()
    test_logits_list = []
    with torch.no_grad():
        with autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(AMP_DTYPE != torch.float32)):
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                test_logits_list.append(outputs.logits.float().cpu().numpy())
    deberta_test_logits = np.concatenate(test_logits_list, axis=0)
    del model, optimizer, scheduler, scaler, ema_full, train_loader, full_ds
    torch.cuda.empty_cache(); gc.collect()

# Convert DeBERTa logits to probs
deberta_test_probs = scipy_softmax(deberta_test_logits / LEARNED_TEMPERATURE, axis=1)

# ============================================================
# PART 2: Retrieval Inference
# ============================================================
print("\nRunning retrieval inference...")
torch.cuda.empty_cache(); gc.collect()

retriever_path = Path("retriever_model")
if retriever_path.exists():
    st_model = SentenceTransformer(str(retriever_path), device=device)
else:
    print("  No saved retriever — using pretrained model (lower quality)")
    st_model = SentenceTransformer(RETRIEVER_MODEL, device=device)

retrieval_test_probs = retrieval_probs(st_model, test_text, label_descs_ordered)
del st_model; torch.cuda.empty_cache(); gc.collect()
print(f"  Retrieval inference done. Shape: {retrieval_test_probs.shape}")

# ============================================================
# PART 3: Ensemble
# ============================================================
print(f"\nEnsembling with α={ENSEMBLE_ALPHA:.2f}...")
test_combined = ENSEMBLE_ALPHA * deberta_test_probs + (1 - ENSEMBLE_ALPHA) * retrieval_test_probs

# Apply family filter
if USE_FAMILY_FILTER:
    for i, ic in enumerate(test_is_correct):
        if ic == 1:
            test_combined[i, ~true_mask] = 0.0
        elif ic == 0:
            test_combined[i, ~false_mask] = 0.0

test_top3 = top3_from_logits(test_combined)

# ============================================================
# Format & Save Submission
# ============================================================
predictions = []
for top3_idx in test_top3:
    labels_str = " ".join([id2lab[j] for j in top3_idx])
    predictions.append(labels_str)

submission = pd.DataFrame({
    "row_id": test_df["row_id"],
    "Category:Misconception": predictions
})

submission.to_csv("submission.csv", index=False)
print(f"\n✅ Submission saved: {len(submission)} rows")
print(submission)

# Show detailed test predictions
for i, row in test_df.iterrows():
    top3_labels = [id2lab[j] for j in test_top3[i]]
    print(f"\nTest {i}: Q={str(row['QuestionText'])[:80]}...")
    print(f"  Answer: {row['MC_Answer']}")
    print(f"  DeBERTa top-1: {id2lab[deberta_test_probs[i].argmax()]} (p={deberta_test_probs[i].max():.4f})")
    print(f"  Retrieval top-1: {id2lab[retrieval_test_probs[i].argmax()]} (p={retrieval_test_probs[i].max():.4f})")
    print(f"  Ensemble top-3: {top3_labels}")

# Cleanup
torch.cuda.empty_cache()
gc.collect()

## Cell 11 — Ablation Study

Expected cumulative MAP@3 improvements from adding each component:

| Technique | Expected Δ MAP@3 | Rationale |
|-----------|:-:|-----------|
| **Vanilla DeBERTa-v3-large** | Baseline | Strong pretrained encoder with disentangled attention |
| + **GroupKFold (no leakage)** | Honest CV | Prevents inflated scores from repeated QuestionIds in train+val |
| + **Focal Loss (γ=2) + Class Weights** | +2–4% | Directly addresses up to 740:1 class imbalance |
| + **Layerwise LR Decay (rate=0.95)** | +0.5–1.0% | Preserves pretrained representations in lower layers |
| + **R-Drop (α=0.5)** | +0.5–1.5% | Consistency regularization via dual dropout forward passes |
| + **AWP (lr=0.1, eps=1e-4)** | +1.0–2.0% | Adversarial weight perturbation → flatter loss surface |
| + **EMA (decay=0.999)** | +0.3–0.8% | Exponential weight averaging → smoother final weights |
| + **bf16 Mixed Precision** | ~0% (speed ↑) | 2× throughput on H100; no accuracy loss versus fp32 |
| + **Multi-Seed Ensemble (2×)** | +0.5–1.0% | Reduces initialization variance; seeds = [42, 123] |
| + **BGE-Large Retrieval Model** | +1–3% | Captures semantic label similarity the classifier misses |
| + **Two-Model Ensemble (α-tuned)** | +1–3% | Diversity-driven error reduction via OOF-learned α |
| + **Temperature Scaling** | +0.2–0.5% | Calibrated probabilities → better top-3 ranking |
| + **Family Filter** | +0.5–1.5% | Hard constraint from known True_*/False_* answer structure |

> **Note**: Ablation chart scores below are indicative **placeholders** — actual gains depend on dataset splits and hyperparameter interactions. Update after a full training run.  
> Cumulative expected improvement: **+8–15% MAP@3** over vanilla fine-tuning.


In [ ]:
# ============================================================
# ABLATION VISUALIZATION
# ============================================================

ablation_techniques = [
    "Vanilla DeBERTa-v3-large",
    "+ GroupKFold",
    "+ Focal Loss + Class Weights",
    "+ Layerwise LR Decay",
    "+ R-Drop",
    "+ AWP",
    "+ EMA",
    "+ Multi-Seed (2×)",
    "+ BGE-Large Retrieval",
    "+ Two-Model Ensemble",
    "+ Temperature Scaling",
    "+ Family Filter",
]

# Placeholder scores (indicative only — update after a full training run)
ablation_scores = [0.62, 0.60, 0.66, 0.67, 0.685, 0.705, 0.715, 0.725, 0.74, 0.77, 0.78, 0.80]

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(ablation_techniques)))
bars = ax.barh(ablation_techniques[::-1], ablation_scores[::-1], color=colors[::-1], edgecolor="black", alpha=0.85)
ax.set_xlabel("MAP@3 Score", fontsize=12)
ax.set_title("Ablation: Cumulative Impact of Each Technique", fontsize=14, fontweight="bold")
ax.set_xlim(0.55, 0.85)

for bar, score in zip(bars, ablation_scores[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f"{score:.3f}", va="center", fontsize=10, fontweight="bold")

ax.axvline(ablation_scores[0], color="gray", linestyle="--", alpha=0.4, label="Baseline")
ax.axvline(ablation_scores[-1], color="green", linestyle="--", alpha=0.4, label="Final")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total improvement: +{ablation_scores[-1] - ablation_scores[0]:.3f} MAP@3")
print(f"Relative improvement: +{(ablation_scores[-1] - ablation_scores[0])/ablation_scores[0]*100:.1f}%")
print("(Scores are placeholders — run full pipeline to get actual values)")


In [ ]:
# ============================================================
# SAVE RESULTS SUMMARY
# ============================================================

import datetime

summary_lines = [
    "=" * 60,
    "  TWO-MODEL ENSEMBLE PIPELINE — RESULTS SUMMARY",
    f"  Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "=" * 60,
    "",
    f"Mode: {'QUICK_RUN' if QUICK_RUN else 'FULL_RUN'}",
    "",
    "--- MODEL A: DeBERTa-v3-large ---",
    f"  Model: {CLASSIFIER_MODEL}",
    f"  Loss: {'FocalLoss(γ=' + str(FOCAL_GAMMA) + ')' if USE_FOCAL_LOSS else 'CE'}",
    f"  Precision: {'bf16' if USE_BF16 else 'fp32'}",
    f"  Folds: {N_FOLDS} | Seeds: {SEEDS} | Epochs: {EPOCHS}",
    f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}",
]
if 'fold_scores' in dir() and fold_scores:
    summary_lines.append(f"  Fold MAP@3: {[round(s, 5) for s in fold_scores]}")
    summary_lines.append(f"  Mean fold MAP@3: {np.mean(fold_scores):.6f}")
if 'deberta_map3' in dir():
    summary_lines.append(f"  DeBERTa OOF MAP@3: {deberta_map3:.6f}")
if 'LEARNED_TEMPERATURE' in dir():
    summary_lines.append(f"  Learned temperature: {LEARNED_TEMPERATURE:.4f}")

summary_lines += [
    "",
    "--- MODEL B: Sentence Transformer ---",
    f"  Model: {RETRIEVER_MODEL}",
    f"  Loss: MultipleNegativesRankingLoss",
    f"  Epochs: {RETRIEVER_EPOCHS} | Batch: {RETRIEVER_BATCH}",
]
if 'retrieval_oof_map3' in dir():
    summary_lines.append(f"  Retrieval OOF MAP@3: {retrieval_oof_map3:.6f}")

summary_lines += [
    "",
    "--- ENSEMBLE ---",
]
if 'ENSEMBLE_ALPHA' in dir():
    summary_lines.append(f"  Alpha: {ENSEMBLE_ALPHA:.2f}")
if 'ensemble_map3' in dir():
    summary_lines.append(f"  ENSEMBLE MAP@3: {ensemble_map3:.6f}")

if 'oof_top3_labels' in dir() and oof_top3_labels[0] is not None:
    top1_acc = sum(1 for i in range(len(train_df)) if oof_top3_labels[i] and oof_top3_labels[i][0] == train_df.iloc[i]["label_str"]) / len(train_df)
    top3_acc = sum(1 for i in range(len(train_df)) if oof_top3_labels[i] and train_df.iloc[i]["label_str"] in oof_top3_labels[i][:3]) / len(train_df)
    summary_lines += [f"  Top-1 Accuracy: {top1_acc:.4f}", f"  Top-3 Accuracy: {top3_acc:.4f}"]

summary_lines += [
    "",
    "--- TECHNIQUES ---",
    f"  Focal Loss: {USE_FOCAL_LOSS} | AWP: {USE_AWP} | R-Drop: {USE_RDROP} | EMA: {USE_EMA}",
    f"  Layerwise LR: {USE_LAYERWISE_LR_DECAY} | Family Filter: {USE_FAMILY_FILTER}",
    f"  bf16: {USE_BF16} | Temperature: {USE_TEMPERATURE_SCALING}",
    "",
    "--- OUTPUT FILES ---",
    "  submission.csv, misconception_kb.json, fold_models/, retriever_model/",
    "=" * 60,
]

summary_text = "\n".join(summary_lines)
Path("results_summary.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)

## Summary & Conclusion

### Two-Model Ensemble Pipeline

| Component | Choice | Why |
|-----------|--------|-----|
| **Model A** | `microsoft/deberta-v3-large` (65-class) | SOTA discriminative encoder with disentangled attention (He et al., 2023) |
| **Model B** | `BAAI/bge-large-en-v1.5` (retrieval) | Semantic similarity captures confusable misconceptions |
| **Loss** | Focal Loss (γ=2) + inverse-frequency class weights | Addresses up to 740:1 class imbalance directly |
| **Precision** | bf16 mixed precision; GradScaler disabled for bf16 | H100 native — 2× throughput, no accuracy loss |
| **Ensemble** | α × DeBERTa + (1-α) × Retrieval | OOF grid search (α ∈ [0.0…1.0, step 0.05]) maximizes diversity |
| **CV Strategy** | GroupKFold(5) by QuestionId × 2 seeds | No leakage across 15 unique QuestionIds; robust estimation |
| **Regularization** | AWP + R-Drop + EMA + Layerwise LR Decay | Multi-pronged: adversarial + consistency + averaging + hierarchical LR |
| **Post-processing** | Family Filter + Temperature Scaling | Hard domain constraints + calibrated probabilities |
| **Explainability** | LLM KB (Qwen2.5-7B, 4-bit NF4) + Teacher Feedback | Pedagogical tool for teachers; skipped in QUICK_RUN |

### Key Design Decisions

1. **Two-model diversity** is the biggest contributor — DeBERTa excels at common/discriminative classes; retrieval generalizes better on rare/confusable ones
2. **Focal Loss** with inverse-frequency weights directly combats extreme class imbalance (~40% of data is `True_Correct:NA`)
3. **AWP ordering fix**: adversarial step runs *before* `scaler.unscale_()` so all gradients remain consistently scaled
4. **Best-fold saving**: only the best-MAP@3 fold checkpoint is kept per seed to avoid filling disk with suboptimal models
5. **OOF-based α tuning** on the ensemble weight prevents overfit to any single model's bias
6. **QUICK_RUN toggle**: single flag that auto-overrides folds/seeds/epochs/AWP/R-Drop/explainer for smoke tests

---

### Full Reference List

1. He, P., Gao, J., & Chen, W. (2023). *DeBERTaV3: Improving DeBERTa using ELECTRA-Style Pre-Training.* ICLR 2023. [arXiv:2111.09543](https://arxiv.org/abs/2111.09543)
2. Wu, D., Xia, S.-T., & Wang, Y. (2020). *Adversarial Weight Perturbation Helps Robust Generalization.* NeurIPS 2020. [arXiv:2004.05884](https://arxiv.org/abs/2004.05884)
3. Wu, X., et al. (2021). *R-Drop: Regularized Dropout for Neural Networks.* NeurIPS 2021. [arXiv:2106.14448](https://arxiv.org/abs/2106.14448)
4. Polyak, B. T. & Juditsky, A. B. (1992). *Acceleration of Stochastic Approximation by Averaging.* SIAM J. Control and Optimization, 30(4), 838–855.
5. Lin, T.-Y., et al. (2017). *Focal Loss for Dense Object Detection.* ICCV 2017. [arXiv:1708.02002](https://arxiv.org/abs/1708.02002)
6. Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* EMNLP 2019. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
7. Xiao, S., et al. (2024). *C-Pack: Packaged Resources for General Chinese Embeddings.* SIGIR 2024. [arXiv:2309.07597](https://arxiv.org/abs/2309.07597)
8. Henderson, M., et al. (2017). *Efficient Natural Language Response Suggestion for Smart Reply.* [arXiv:1705.00652](https://arxiv.org/abs/1705.00652) — MultipleNegativesRankingLoss basis
9. Clark, K., et al. (2020). *ELECTRA: Pre-training Text Encoders as Discriminators Rather Than Generators.* ICLR 2020. [arXiv:2003.10555](https://arxiv.org/abs/2003.10555) — Layerwise LR Decay
10. Guo, C., et al. (2017). *On Calibration of Modern Neural Networks.* ICML 2017. [arXiv:1706.04599](https://arxiv.org/abs/1706.04599) — Temperature Scaling
11. Lakshminarayanan, B., et al. (2017). *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles.* NeurIPS 2017. [arXiv:1612.01474](https://arxiv.org/abs/1612.01474)
12. Loshchilov, I. & Hutter, F. (2017). *SGDR: Stochastic Gradient Descent with Warm Restarts.* ICLR 2017. [arXiv:1608.03983](https://arxiv.org/abs/1608.03983) — Cosine LR Schedule
13. Szegedy, C., et al. (2016). *Rethinking the Inception Architecture for Computer Vision.* CVPR 2016. [arXiv:1512.00567](https://arxiv.org/abs/1512.00567) — Label Smoothing
14. Vaswani, A., et al. (2017). *Attention Is All You Need.* NeurIPS 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762) — Transformer foundation
15. Qwen Team (2024). *Qwen2.5 Technical Report.* [arXiv:2412.15115](https://arxiv.org/abs/2412.15115)
16. Dettmers, T., et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* NeurIPS 2023. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314) — 4-bit NF4 quantization (BitsAndBytes)
17. Eedi & The Learning Agency (2025). *MAP – Charting Student Math Misunderstandings.* [Kaggle](https://kaggle.com/competitions/map-charting-student-math-misunderstandings)
